# 13 · Ⅱ-02 데이터 가공과 핵심 속성 추출 — 교과서 70~83쪽 전체
이 노트북 한 권에 **교과서 70~83쪽의 본문 · 옆 설명 · 실습 코드 ❶~⓮ · 읽을거리(상자그림)** 를 모두 담았습니다.
마지막에는 **80~82쪽 활동(건강검진 데이터)** 을 추가 활동으로 붙였습니다.

맨 먼저 **파일 → 드라이브에 사본 저장**을 하세요(내 드라이브에 저장돼야 고친 내용이 남습니다).
`____` 가 보이면 거기가 문제입니다. 빈칸 **바로 위 💡 힌트**를 읽고 채운 뒤 셀을 실행하세요(Shift + Enter).
교과서는 `files.upload()` 로 파일을 올리지만, 이 노트북은 **선생님 저장소에서 바로 읽어** 올릴 파일이 없습니다.

**학습 목표** (70쪽)
- 결측치와 이상치의 유무를 파악할 수 있다.
- 문제 해결에 필요한 속성이 무엇인지 선별할 수 있다.
- 데이터를 문제 해결에 적합한 형태로 전처리할 수 있다.

| 순서 | 내용 | 쪽 |
|---|---|---|
| 0 | 생각 열기 — 생일 파티 명단 | 70 |
| 1 | 전처리의 필요성과 네 가지 유형 (변환 · 통합 · 정제 · 축소) | 71 |
| 2 | 실습 ❶~⓫ — 의료비 데이터 전처리 | 72~77 |
| 3 | 실습 ⓬~⓮ — 데이터 축소와 핵심 속성(상관계수) | 78~79 |
| 4 | 읽을거리 — 상자그림과 IQR | 83 |
| 5 | **추가 활동** — 건강검진 데이터로 혈색소와 관련 깊은 속성 찾기 | 80~82 |

---
## 0. 생각 열기 (70쪽)
전교 회장 **동동일**과 부회장 **동동이**는 «월별 생일자를 모아 생일 파티를 하겠다»는 공약을 냈습니다.
당선 후 두 사람은 **각자** 전교생에게 개인 정보 동의서를 받고 생년월일을 모았어요.

- 회장한테만 낸 학생 · 부회장한테만 낸 학생 · **두 사람 모두에게 낸 학생** · 아예 안 낸 학생이 섞여 있습니다.
- 그래서 **중복**도 있고 **빠진 응답**도 있고, 적은 **형태도 제각각**(070103 · 07/01/03 · 2007년 6월 15일 …)입니다.

> 🤖 «즉시 사용할 수 없는 형태로 데이터가 수집되는 일은 흔해. 수집된 데이터를 본격적으로 사용하는 단계(**본처리**)에 앞서 무엇을 해야 할까?»

이 «앞서 하는 일»이 바로 **전처리**입니다.

---
## 1. 데이터 전처리의 필요성과 유형 (71쪽)
데이터를 모으다 보면 특정 항목이 **없거나(결측치)**, **중복**되거나, **표현에 문제가 있는** 경우가 자주 생깁니다.
- **이상치** — 정상 범주에서 **벗어난** 값 (예: 키 973cm, 2월 31일)
- **결측치** — 기록상 **빈칸**인 값 (예: 지하철역 이용자 수에서 특정 날짜가 누락, 생년월일에서 «일»이 빠짐)

수집한 데이터를 목적에 맞게 쓰려면 대부분 **사전 가공**이 필요한데, 이것이 **전처리**입니다.
목적과 수집 상황에 따라 전처리 절차와 결과는 달라질 수 있습니다.

| 유형 | 하는 일 | 교과서 예 |
|---|---|---|
| **데이터 변환** | 표현 형태를 일관성 있게 통일 | 생년월일 070103 · 07/01/03 → 20070103 |
| **데이터 통합** | 여러 출처의 데이터를 하나로 | 백화점.csv(귤 35) + 온라인 판매.csv(귤 150) → 매출 합계(185) |
| **데이터 정제** | 이상치 · 결측치를 수정하거나 제거 | 키 772(이상치), 키 빈칸(결측치) |
| **데이터 축소** | 불필요한 데이터 제거 · 요약해 크기 줄이기 | «고객의 생일은 마트 이용 횟수 예측에 영향이 없다» → 생일 열 삭제 |

아래 네 셀은 네 가지 전처리를 **아주 작은 표**로 먼저 해 보는 것입니다. 차례로 실행하며 «전 → 후»를 비교하세요.

In [ ]:
import pandas as pd

# ① 변환 — 생년월일 표현 통일
생일 = pd.DataFrame({'생일': ['070103', '07/01/03', '2007/04/07', '2007년 6월 15일']})
생일['통일'] = ['20070103', '20070103', '20070407', '20070615']   # 사람이 규칙을 정해 통일한 결과
생일

In [ ]:
# ② 통합 — 두 판매처의 기록을 하나로
백화점 = pd.DataFrame({'품목': ['귤'], '수량(박스)': [35]})
온라인 = pd.DataFrame({'품목': ['귤'], '수량(박스)': [150]})
합계 = pd.concat([백화점, 온라인]).groupby('품목', as_index=False).sum()
합계

In [ ]:
# ③ 정제 — 이상치(772)와 결측치(빈칸) 찾기
키 = pd.DataFrame({'사람': ['A', 'B', 'C', 'D', 'E'], '키': [186, 155, 772, None, 166]})
print(키)
print('결측치 개수 :', 키['키'].isnull().sum())
print('정상 범위(100~230)를 벗어난 값 :', 키[(키['키'] < 100) | (키['키'] > 230)]['키'].tolist())

In [ ]:
# ④ 축소 — 목표(연중 마트 이용 횟수 예측)와 관계없는 열 빼기
고객 = pd.DataFrame({'거주지~마트 거리': [1.2, 3.5], '생일': ['0103', '0615'], '가족 구성원 수': [4, 2], '연중 마트 이용 횟수': [52, 18]})
고객 = 고객.drop('생일', axis=1)
고객

<details><summary>🧐 30초 개념 정리 (71쪽) — 눌러서 답 보기</summary>

1. 이상치는 정상 범주에서 벗어난 값이고, 결측치는 기록상 빈칸인 값이다. → **O**
2. 데이터 ( **정제** )는 수집된 데이터의 이상치 또는 결측치를 수정하거나 제거하는 작업이다.

</details>

---
## 2. 데이터 전처리 실습 (72~77쪽)
> 전처리를 **대량으로** 하려면 프로그래밍을 통한 **자동화**가 필수입니다.

**상황** — 공공의료기관에 근무하는 **나산출** 씨는 개인의 **연간 의료비 지출을 예측하는 모델**을 만들려고 데이터를 받았습니다.
(계약 번호 · 나이 · 성별 · BMI · 자녀 수 · 흡연 여부 · 연락처 · 의료비)  그런데 문제가 보입니다.
- 파일이 **두 개로 나뉘어** 있다
- **빈칸**이 있다
- **288세** 같은 이상한 데이터가 있다
- **연락처**처럼 의료비 예측과 관련 없는 항목이 있다
- 성별이 ‘여’, ‘남’, ‘남자’, ‘여자’ 등 **여러 가지로 표현**돼 있다

### 과정 ❶ 판다스 라이브러리 사용 준비하기
> 📌 **판다스(pandas)** — 데이터 분석에 쓰는 라이브러리. 유연한 데이터 구조와 도구를 제공한다.
> `import pandas as pd` → 표 형태 데이터를 다루는 판다스를 **별칭 pd** 로 쓰겠다는 준비.

In [ ]:
import pandas as pd

### 과정 ❷ 데이터 사용 준비하기
교과서는 `files.upload()` 로 파일 두 개를 올립니다(아래 접은 칸). 우리는 **선생님 저장소 주소에서 바로** 읽습니다.

> 📌 **데이터프레임(dataframe)** — 행과 열로 이루어진 **표** 형태의 자료 구조. 행은 개별 관찰값, 열은 그 관찰값의 특성(변수).
>
> | 확장명 | 파일 읽기 함수 |
> |---|---|
> | csv | `read_csv('파일')` |
> | xls 또는 xlsx | `read_excel('파일')` |

<details><summary>교과서 방식(시험 대비) — 눌러서 보기</summary>

```python
from google.colab import files   # 01 구글 코랩의 files 모듈을 쓰기 위한 명령
files.upload()                   # 02 «파일 선택» 버튼이 나타남 → 두 파일을 한꺼번에 선택
df1 = pd.read_csv('file1.csv')   # 03 file1.csv 를 데이터프레임으로 읽어 df1 에 할당
df2 = pd.read_csv('file2.csv')   # 04 file2.csv 를 데이터프레임으로 읽어 df2 에 할당
```
</details>

In [ ]:
주소 = 'https://raw.githubusercontent.com/richee-pc/AI_cs/main/data/'
df1 = pd.read_csv(주소 + 'file1.csv')
df2 = pd.read_csv(주소 + 'file2.csv')
df1

In [ ]:
df2

**살펴보기** — df1 의 ‘성별’은 «남 · 여», df2 는 «여 · 여자 · 남자». df2 의 11번 계약은 **BMI 가 NaN(빈칸)**, df1 의 3번은 **나이 288**.

### 과정 ❸ 데이터 변환하기
> 📌 **replace()** — 속성 전체에서 특정 값을 다른 값으로 **일괄** 바꾼다. `{'바꿀 값': '새 값'}` 사전 형태로 준다.

💡 «남자» → «남», «여자» → «여».

In [ ]:
df2['성별'] = df2['성별'].replace({'남자': '남', '여자': ____})
df2

### 과정 ❹ 데이터 통합하기
> 📌 **concat()** — 행 방향 또는 열 방향으로 데이터를 결합. **기본값은 행 방향**(아래로 이어 붙이기).

💡 합칠 데이터프레임들을 **리스트**로 → `[df1, df2]`

In [ ]:
df = pd.concat(____)
df

통합할 때는 **중복**과 **단위 불일치**가 없는지 확인해야 합니다. 단위 불일치는 없으니 중복을 봅니다.
위 표에서 **계약번호 6** 이 두 번 나오지요?

### 과정 ❺ 데이터 중복 확인하기
> 📌 **duplicated()** — 값마다 중복이면 True, 아니면 False
> 📌 **any()** — 하나라도 True 가 있으면 True

데이터가 많을 때는 눈으로 찾을 수 없으니 이 두 함수를 이어 씁니다.

In [ ]:
df['계약번호'].duplicated().any()

### 과정 ❻ 중복 데이터 삭제하기
`drop_duplicates()` 는 한 행만 남기고 나머지 중복 행을 지웁니다.
`keep='first'` 는 «중복된 행들 중 **첫 행**이 정상이라 가정하고 남겨라»는 뜻입니다.

In [ ]:
print('삭제 전 :', df.shape)
df = df.drop_duplicates(subset='계약번호', keep='first')
print('삭제 후 :', df.shape)

### 3 데이터 정제 — 결측치와 이상치를 처리하는 두 방법 (75쪽 표 Ⅱ-5)
| 방법 | 설명 |
|---|---|
| **제거** | 값을 신뢰할 수 없으면 그 값 또는 그 행 자체를 삭제 |
| **대체** | 다시 수집할 수 있으면 새 값으로, 어렵다면 **최빈값**이나 **평균** 같은 추정치로 대체 |

> 🤖 «소속 학년 · 반 같은 **범주형** 데이터는 **최빈값**으로, 키 · 손 크기 · BMI 같은 **연속형** 데이터는 **평균**으로 대체할 수 있어.»

상황을 종합해 제거와 대체 중 알맞은 것을 고릅니다. **결측치나 이상치 자체가 의미 있거나 고칠 필요가 없으면 함부로 지우거나 바꾸지 않습니다.**
(이 예제에서는 지워도 괜찮다고 가정하고 **행 단위로 삭제**합니다.)

### 과정 ❼ 결측치 찾기
> 📌 **isnull()** — 빈칸이면 True, 아니면 False
> 📌 **sum()** — 속성마다 값을 모두 더함. True 는 1, False 는 0 으로 계산 → 열마다 «빈칸 개수»

> 🤖 «`df.info()` 의 **Non-Null** 열(비어 있지 않은 값의 개수)에서도 결측치를 짐작할 수 있어.»

In [ ]:
df.____().sum()

In [ ]:
df.info()

### 과정 ❽ 결측치 삭제하기
`dropna()` 는 결측치가 있는 **행**을 삭제합니다. `shape` (행의 개수, 열의 개수)로 줄어든 것을 확인합니다.

In [ ]:
print(df.shape)
df = df.dropna()
print(df.shape)

### 과정 ❾ 이상치 탐색하기 — 상자그림
> 📌 `import matplotlib.pyplot as plt` — 시각화 라이브러리 matplotlib 을 **plt** 라는 별칭으로 준비
> 🤖 «코랩에서는 `plt.show()` 가 없어도 그림이 보이지만, **원칙적으로는 적는 것이 좋아.**»
> 🤖 «matplotlib 으로 막대그래프 · 산점도 · 꺾은선그래프 · 히스토그램 · 원그래프 등 여러 가지를 그릴 수 있어.»

💡 상자그림 함수 이름은 `boxplot`

In [ ]:
import matplotlib.pyplot as plt
plt.____(df['나이'])
plt.title('age')
plt.show()

상자 바깥에 **200세가 넘는 점 하나**가 보입니다. (왜 그 점이 «바깥»으로 그려지는지는 아래 **4. 상자그림 읽을거리**에서!)

### 과정 ❿ 이상치 제거 및 재시각화하기
200세를 최대 수명으로 보고, ‘나이’가 **200 이하**인 행만 남깁니다.
💡 «이하»는 `<=`

In [ ]:
condition = df['나이'] ____ 200     # 01 조건: 나이가 200 이하
df = df[condition]                  # 02 조건이 True 인 행들만으로 df 를 다시 구성
plt.boxplot(df['나이'])             # 03 다시 상자그림
plt.show()                          # 04 화면에 출력
print(df.shape)

### 과정 ⓫ 전처리를 마친 데이터 저장하기
`to_csv()` 로 새 CSV 파일로 저장합니다. 계약 번호가 있으니 **index(행 번호)는 저장하지 않도록** `index=False`.
> 🤖 «왼쪽 📁 **파일** 창을 새로고침하면 저장된 파일을 볼 수 있어. 다운로드 · 이름 바꾸기 · 삭제 · 경로 복사도 거기서.»

> ⚠️ 교과서 77쪽 본문은 «총 9개의 데이터»라고 적었지만, 실제로 실행하면 **10개**입니다(78쪽 결과 표도 10행). 바로 위 `shape` 를 확인해 보세요.

In [ ]:
df.to_csv('combined_file.csv', index=False)

---
## 3. 데이터 축소와 핵심 속성 추출 (78~79쪽)
**데이터 축소** — 목표와 관련 없는 데이터를 제거해 **핵심 속성만** 남기는 과정. 분석 정확도를 지키거나 높이고, 처리 시간을 줄이고, 저장 공간을 아낍니다.

- **연락처** → 의료비와 상관없음
- **계약 번호** → 중복을 거르는 데 썼지만 의료비 예측에는 필요 없음
- 나이 · 성별 · BMI · 자녀 수 · 흡연 여부 → 건강에 영향을 줄 수 있으니 **남김**

### 과정 ⓬ 데이터 축소하기
💡 `axis=1` 은 **열** 단위로 선택해 삭제하라는 뜻 (`axis=0` 이면 행).

In [ ]:
df = df.drop('연락처', axis=1)
df = df.drop('계약번호', axis=____)
df

### 과정 ⓭ 핵심 속성 추출을 위한 시각화
**뚜렷한 연관성이 보이면 핵심 속성**, 아무 연관성이 없으면 핵심 속성이 아닙니다. 두 변수의 관계는 **산점도**로 봅니다.
> 📌 **산점도** — 두 변수 사이의 관계를 점으로 나타낸 그래프

In [ ]:
plt.scatter(df['나이'], df['의료비'])
plt.xlabel('age')
plt.ylabel('cost')
plt.show()

나이와 의료비 사이의 연관성은 그림만으로는 찾기 어렵습니다. 그래서 연관성을 **수치로** 나타내는 **상관계수**를 씁니다.

### 과정 ⓮ 변수 간 상관관계 분석
> 📌 **상관계수(correlation coefficient)** — 두 변수 사이의 **선형 관계의 강도**를 나타내는 척도. **−1 이상 1 이하**, 절댓값이 클수록 뚜렷한 선형 관계.
> - 양수 → 함께 증가하거나 함께 감소 · 음수 → 한쪽이 늘 때 다른 쪽은 줄어듦
>
> 📌 **연속형 데이터** — 숫자로 측정된 데이터(키, 몸무게, 과목 점수)
> 📌 **범주형 데이터** — 몇 개의 항목으로 나뉜 데이터(성별 남/여, 흡연 여부 ◦/×)

`corr(numeric_only=True)` — **연속형(숫자) 속성끼리만** 상관계수를 구합니다.

In [ ]:
corr = df.corr(numeric_only=True)
corr

In [ ]:
# (넓혀 보기) 상관계수를 색으로 — 히트맵
import seaborn as sns
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

**해석** — 의료비와의 상관: **BMI 약 0.75** > 자녀 수 약 0.19 > 나이 약 0.10. BMI 가 의료비와 가장 큰 연관을 보입니다.

> 🤖 «실습 데이터는 **적은 수의 표본**만 썼으니 **함부로 일반화하지 않도록** 주의해. ‘BMI가 나이보다 의료비 지출과 더 큰 연관이 있다’를 일반적인 사실로 받아들이면 안 돼.»
> 그리고 상관은 **함께 움직인다**는 뜻일 뿐, **원인과 결과**는 아닙니다.

---
## 4. 읽을거리 — 상자그림(Box Plot)과 IQR (83쪽)
이상치는 정확한 기준을 세우기 어려워 결측치보다 찾기 어렵습니다. 오류 없이 찾으려면 그 분야의 **전문성**이 필요하고, 기준점이 없으면 완전히 **자동화하기도 어렵습니다**. 이럴 때 쓰는 도구가 **상자그림**입니다.

- 상자그림은 데이터의 **0% · 25%(Q1) · 50%(중앙값) · 75%(Q3) · 100%** 지점이 드러나게 그립니다.
- **IQR** = Q3 − Q1 (가운데 50% 데이터가 퍼진 폭)
- **Q1 − 1.5×IQR 보다 작거나, Q3 + 1.5×IQR 보다 큰** 값 → **이상치** (상자 수염 바깥의 **점**)

아래 셀은 file1 의 나이로 **직접 계산**해서, 상자그림이 왜 288을 점으로 그렸는지 확인합니다.

In [ ]:
나이 = pd.read_csv(주소 + 'file1.csv')['나이']
Q1 = 나이.quantile(0.25)
Q3 = 나이.quantile(0.75)
IQR = Q3 - Q1
아래 = Q1 - 1.5 * IQR
위 = Q3 + 1.5 * IQR
print('나이 값 :', sorted(나이.tolist()))
print(f'Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}')
print(f'정상 범위 : {아래} ~ {위}')
print('이상치 :', 나이[(나이 < 아래) | (나이 > 위)].tolist())

plt.boxplot(나이)
for y, t in ((Q1, 'Q1'), (나이.median(), 'median'), (Q3, 'Q3'), (위, 'Q3+1.5*IQR')):
    plt.axhline(y, ls=':', lw=1); plt.text(1.12, y, t)
plt.show()

**해 보기** — 위 셀의 288 을 60 으로 바꿔도 이상치로 나올까요? 직접 바꿔 보고 이유를 적어 보세요.

> 적어 보기:

<details><summary>🧐 30초 개념 정리 · 진단하기 (82쪽) — 눌러서 보기</summary>

- 데이터 변환 · 통합 · 정제 · 축소를 수행할 수 있는가? (상 · 중 · 하)
- 수집된 데이터를 가공하여 핵심 속성을 추출할 수 있는가? (상 · 중 · 하)
- 상관계수가 −0.9 인 두 변수는 관계가 약하다 → **X** (절댓값이 크면 강한 관계, 음수는 반대 방향)

</details>

---
## 5. 추가 활동 — 생명과학과 인공지능 : 혈색소와 가장 관련 깊은 속성 찾기 (80~82쪽)
**혈색소**(헤모글로빈)는 산소와 이산화탄소를 운반하는 적혈구의 주요 성분입니다.
공공데이터포털의 «국민건강보험공단_건강검진정보»로 **혈색소와 가장 연관성이 높은 속성**을 찾아봅니다.

> 📌 **EDA(탐색적 데이터 분석)** — 데이터의 기본 특성을 시각적 · 통계적으로 파악하는 초기 분석 과정
> 🤖 교과서는 100만 명 파일(약 95MB)을 구글 드라이브의 `02/02` 폴더에 올리고 `drive.mount` 로 연결합니다(아래 접은 칸). 여기서는 같은 데이터에서 **3만 명을 무작위로 뽑은 표본**을 바로 읽습니다.

<details><summary>교과서 방식 — 드라이브 연결</summary>

```python
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/My Drive/02/02/국민건강보험공단_건강검진정보.csv', encoding='cp949')
```
</details>

### 단계 1 · 2 — 확보하고 살펴보기

In [ ]:
hc = pd.read_csv(주소 + 'health_checkup_30k.csv')
hc.head()

💡 데이터프레임의 정보(열 · 자료형 · Non-Null 개수)를 출력하는 함수는?

In [ ]:
hc.____()

### 단계 3 · 데이터 전처리
결손치유무 · 치아마모증율 · 제3대구치(사랑니)이상은 **Non-Null 이 0** — 열 전체가 비어 있습니다.

In [ ]:
hc.isnull().sum()

In [ ]:
# 건강과 직접 관련이 없는 속성
hc = hc.drop(['기준년도', '가입자일련번호', '시도코드'], axis=1)
# 열 전체가 결측치인 속성
hc = hc.drop(['결손치유무', '치아마모증율', '제3대구치(사랑니)이상'], axis=1)
# 상관계수를 적용하기 어려운 범주형 속성
hc = hc.drop(['성별', '청력(좌)', '청력(우)', '흡연상태', '음주여부', '구강검진수검여부', '치아우식증유무', '치석'], axis=1)
hc.columns

💡 특정 열의 이상치를 찾는 그림 → 상자그림
> 🤖 «연습을 위해 상자그림을 참고해 **18 이상**을 이상치로 보고 뺐어. 실제 데이터를 전처리할 때는 **충분히 생각하고** 결정을 내려!»

In [ ]:
plt.____(hc['연령대코드(5세 단위)'])
plt.show()
hc = hc[hc['연령대코드(5세 단위)'] < 18]
plt.boxplot(hc['연령대코드(5세 단위)'])
plt.show()

### 단계 4 · 상관계수를 도출하고 핵심 속성 찾기
💡 상관계수를 구하는 함수는?

In [ ]:
corr = hc.____()
corr = corr['혈색소']
print('— 양의 상관 상위 —')
print(corr.sort_values(ascending=False).head())
print('— 음의 상관 상위 —')
print(corr.sort_values(ascending=True).head())

**정리해 보기**
1. 혈색소와 양의 상관이 큰 속성 셋: ______ (교과서: 신장 0.48 · 체중 0.45 · 허리둘레 0.35)
2. 음의 상관: ______ (교과서: HDL콜레스테롤 −0.18 · 연령대 −0.14)
3. «키가 크면 혈색소가 높다»는 원인과 결과일까요? 둘 다에 영향을 주는 **성별**을 떠올려 보세요.

> 적어 보기:

---
### 정답 예시 (막혔을 때만)
- ❸ `'여'` · ❹ `[df1, df2]` · ❼ `isnull` · ❾ `boxplot` · ❿ `<=` · ⓬ `axis=1`
- 추가 활동: `info` · `boxplot` · `corr`